In [ ]:
!pip install -q nltk
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)


True

In [ ]:
import time
import math
import random
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

torch.manual_seed(42)
random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [ ]:
from google.colab import files

uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {DATA_PATH}")


Saving vast_english_french.txt to vast_english_french.txt
Uploaded: vast_english_french.txt


In [ ]:
def parse_pairs(path):
    pairs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if "\t" in line:
                parts = line.split("\t")
            else:
                parts = line.split(",")
            if len(parts) < 2:
                continue
            eng, fra = parts[0].strip(), parts[1].strip()
            if eng and fra:
                pairs.append((eng, fra))
    return pairs

raw_pairs = parse_pairs(DATA_PATH)
print(f"Loaded {len(raw_pairs)} sentence pairs")


pairs = [(fra, eng) for (eng, fra) in raw_pairs]
print("Example (French -> English):", pairs[0])


Loaded 555 sentence pairs
Example (French -> English): ("J'ai froid", 'I am cold')


In [ ]:
def tokenize(s):
    s = s.lower().strip()
    s = re.sub(r"([.!?,\'\"])", r" \1 ", s)
    s = re.sub(r"[^a-zA-Zà-ÿÀ-ß.!?,\'\" ]+", " ", s)
    return [t for t in s.split() if t]

PAD, SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"

def build_vocab(sentences, min_freq=1):
    freq = {}
    for s in sentences:
        for tok in tokenize(s):
            freq[tok] = freq.get(tok, 0) + 1
    itos = [PAD, SOS, EOS, UNK] + [w for w, c in freq.items() if c >= min_freq]
    stoi = {w: i for i, w in enumerate(itos)}
    return stoi, itos

src_sentences = [p[0] for p in pairs]
tgt_sentences = [p[1] for p in pairs]

src_stoi, src_itos = build_vocab(src_sentences)
tgt_stoi, tgt_itos = build_vocab(tgt_sentences)

print("French (source) vocab size:", len(src_itos))
print("English (target) vocab size:", len(tgt_itos))

def encode_sentence(s, stoi, max_len):
    toks = [SOS] + tokenize(s) + [EOS]
    ids = [stoi.get(t, stoi[UNK]) for t in toks][:max_len]
    ids += [stoi[PAD]] * (max_len - len(ids))
    return ids

MAX_LEN_SRC = max(len(tokenize(s)) for s in src_sentences) + 2
MAX_LEN_TGT = max(len(tokenize(s)) for s in tgt_sentences) + 2
print("Max source (French) length:", MAX_LEN_SRC, "| Max target (English) length:", MAX_LEN_TGT)


French (source) vocab size: 1126
English (target) vocab size: 1015
Max source (French) length: 18 | Max target (English) length: 13


In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, src_stoi, tgt_stoi, max_len_src, max_len_tgt):
        self.pairs = pairs
        self.src_stoi = src_stoi
        self.tgt_stoi = tgt_stoi
        self.max_len_src = max_len_src
        self.max_len_tgt = max_len_tgt

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_s, tgt_s = self.pairs[idx]
        src = encode_sentence(src_s, self.src_stoi, self.max_len_src)
        tgt = encode_sentence(tgt_s, self.tgt_stoi, self.max_len_tgt)
        return torch.tensor(src, dtype=torch.long), torch.tensor(tgt, dtype=torch.long)


full_ds = TranslationDataset(pairs, src_stoi, tgt_stoi, MAX_LEN_SRC, MAX_LEN_TGT)
n_val = int(0.2 * len(full_ds))
n_train = len(full_ds) - n_val
train_ds, val_ds = torch.utils.data.random_split(
    full_ds, [n_train, n_val], generator=torch.Generator().manual_seed(42)
)
print(f"Train pairs: {len(train_ds)} | Val pairs: {len(val_ds)}")

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)


Train pairs: 444 | Val pairs: 111


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class Seq2SeqTransformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=128, n_heads=2, n_layers=2,
                 d_ff=256, dropout=0.1, max_len=200, pad_idx=0):
        super().__init__()
        self.d_model = d_model
        self.pad_idx = pad_idx

        self.src_emb = nn.Embedding(src_vocab, d_model, padding_idx=pad_idx)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model, padding_idx=pad_idx)
        self.pos_enc = PositionalEncoding(d_model, max_len)

        self.transformer = nn.Transformer(
            d_model=d_model, nhead=n_heads,
            num_encoder_layers=n_layers, num_decoder_layers=n_layers,
            dim_feedforward=d_ff, dropout=dropout, batch_first=True,
        )
        self.fc_out = nn.Linear(d_model, tgt_vocab)

    def make_pad_mask(self, seq):
        return seq == self.pad_idx

    def forward(self, src, tgt_in):
        src_key_padding_mask = self.make_pad_mask(src)
        tgt_key_padding_mask = self.make_pad_mask(tgt_in)
        tgt_len = tgt_in.size(1)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_len).to(src.device)

        src_emb = self.pos_enc(self.src_emb(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_enc(self.tgt_emb(tgt_in) * math.sqrt(self.d_model))

        out = self.transformer(
            src_emb, tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )
        return self.fc_out(out)

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def estimate_seq2seq_flops(src_len, tgt_len, d_model, n_layers, d_ff):
    enc_self_attn = n_layers * (4 * src_len * d_model**2 + 2 * src_len**2 * d_model
                                 + 2 * src_len * d_model * d_ff * 2)
    dec_self_attn = n_layers * (4 * tgt_len * d_model**2 + 2 * tgt_len**2 * d_model
                                 + 2 * tgt_len * d_model * d_ff * 2)
    dec_cross_attn = n_layers * (2 * tgt_len * src_len * d_model + 2 * tgt_len * d_model**2)
    return enc_self_attn + dec_self_attn + dec_cross_attn


In [ ]:
PAD_IDX = tgt_stoi[PAD]

def train_model(model, train_loader, val_loader, epochs=25, lr=1e-3, verbose=True):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

    train_losses, val_losses = [], []
    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        total_loss, n_batches = 0.0, 0
        for src, tgt in train_loader:
            src, tgt = src.to(device), tgt.to(device)
            tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]

            optimizer.zero_grad()
            logits = model(src, tgt_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1
        avg_train_loss = total_loss / n_batches
        train_losses.append(avg_train_loss)

        model.eval()
        val_loss_sum, val_batches = 0.0, 0
        with torch.no_grad():
            for src, tgt in val_loader:
                src, tgt = src.to(device), tgt.to(device)
                tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
                logits = model(src, tgt_in)
                loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
                val_loss_sum += loss.item()
                val_batches += 1
        avg_val_loss = val_loss_sum / val_batches
        val_losses.append(avg_val_loss)

        if verbose and ((epoch + 1) % 5 == 0 or epoch == 0):
            print(f"  Epoch {epoch+1:3d}/{epochs} | train_loss={avg_train_loss:.4f} | val_loss={avg_val_loss:.4f}")

    training_time = time.time() - start_time
    return train_losses, val_losses, training_time


In [ ]:
def greedy_decode(model, src, max_len, sos_idx, eos_idx, pad_idx):
    model.eval()
    src = src.to(device)
    src_key_padding_mask = model.make_pad_mask(src)
    src_emb = model.pos_enc(model.src_emb(src) * math.sqrt(model.d_model))

    with torch.no_grad():
        memory = model.transformer.encoder(
            src_emb, src_key_padding_mask=src_key_padding_mask
        )

    B = src.size(0)
    ys = torch.full((B, 1), sos_idx, dtype=torch.long, device=device)
    finished = torch.zeros(B, dtype=torch.bool, device=device)

    for _ in range(max_len - 1):
        tgt_emb = model.pos_enc(model.tgt_emb(ys) * math.sqrt(model.d_model))
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(ys.size(1)).to(device)
        with torch.no_grad():
            out = model.transformer.decoder(
                tgt_emb, memory, tgt_mask=tgt_mask,
                memory_key_padding_mask=src_key_padding_mask,
            )
            logits = model.fc_out(out[:, -1, :])
        next_tok = logits.argmax(dim=-1)
        next_tok = torch.where(finished, torch.full_like(next_tok, pad_idx), next_tok)
        ys = torch.cat([ys, next_tok.unsqueeze(1)], dim=1)
        finished = finished | (next_tok == eos_idx)
        if finished.all():
            break
    return ys


def ids_to_tokens(ids, itos, stop_at_eos=True):
    toks = []
    for i in ids:
        tok = itos[i]
        if tok == EOS and stop_at_eos:
            break
        if tok in (PAD, SOS):
            continue
        toks.append(tok)
    return toks


def evaluate_bleu_and_accuracy(model, val_loader, max_len_tgt):
    smoothie = SmoothingFunction().method4
    total, exact_matches, bleu_sum = 0, 0, 0.0
    examples = []

    sos_idx, eos_idx, pad_idx = tgt_stoi[SOS], tgt_stoi[EOS], tgt_stoi[PAD]

    for src, tgt in val_loader:
        preds = greedy_decode(model, src, max_len_tgt, sos_idx, eos_idx, pad_idx)
        for i in range(src.size(0)):
            ref_toks = ids_to_tokens(tgt[i].tolist(), tgt_itos)
            hyp_toks = ids_to_tokens(preds[i].tolist(), tgt_itos)

            bleu = sentence_bleu([ref_toks], hyp_toks, weights=(0.25, 0.25, 0.25, 0.25),
                                  smoothing_function=smoothie)
            bleu_sum += bleu
            is_match = (ref_toks == hyp_toks)
            exact_matches += int(is_match)
            total += 1

            if len(examples) < 8:
                src_toks = ids_to_tokens(src[i].tolist(), src_itos)
                examples.append({
                    "source": " ".join(src_toks),
                    "reference": " ".join(ref_toks),
                    "prediction": " ".join(hyp_toks),
                    "exact_match": is_match,
                    "bleu": bleu,
                })

    seq_accuracy = exact_matches / total
    avg_bleu = bleu_sum / total
    return seq_accuracy, avg_bleu, examples


In [ ]:
D_MODEL, D_FF, EPOCHS = 128, 256, 25
block_options = [1, 2, 4]
head_options = [2, 4]

sweep_results = []
trained_models = {}

for n_layers in block_options:
    for n_heads in head_options:
        print(f"\n=== Config: n_layers={n_layers}, n_heads={n_heads} ===")
        model = Seq2SeqTransformer(
            src_vocab=len(src_itos), tgt_vocab=len(tgt_itos),
            d_model=D_MODEL, n_heads=n_heads, n_layers=n_layers, d_ff=D_FF,
            max_len=max(MAX_LEN_SRC, MAX_LEN_TGT) + 5, pad_idx=PAD_IDX,
        )
        n_params = model.count_params()
        flops = estimate_seq2seq_flops(MAX_LEN_SRC, MAX_LEN_TGT, D_MODEL, n_layers, D_FF)

        train_losses, val_losses, train_time = train_model(
            model, train_loader, val_loader, epochs=EPOCHS, verbose=False
        )
        seq_acc, avg_bleu, examples = evaluate_bleu_and_accuracy(model, val_loader, MAX_LEN_TGT)

        row = {
            "n_layers": n_layers, "n_heads": n_heads,
            "final_train_loss": train_losses[-1], "final_val_loss": val_losses[-1],
            "seq_accuracy": seq_acc, "bleu4": avg_bleu,
            "training_time_sec": train_time,
            "num_params": n_params, "approx_flops": flops,
        }
        sweep_results.append(row)
        trained_models[(n_layers, n_heads)] = (model, examples)

        print(f"  -> train_loss={train_losses[-1]:.4f} | val_loss={val_losses[-1]:.4f} "
              f"| seq_acc={seq_acc:.4f} | BLEU-4={avg_bleu:.4f} | time={train_time:.1f}s | params={n_params:,}")



=== Config: n_layers=1, n_heads=2 ===


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


  -> train_loss=0.6350 | val_loss=4.2520 | seq_acc=0.0090 | BLEU-4=0.1434 | time=4.9s | params=736,759

=== Config: n_layers=1, n_heads=4 ===
  -> train_loss=0.4884 | val_loss=4.2993 | seq_acc=0.0090 | BLEU-4=0.1485 | time=4.5s | params=736,759

=== Config: n_layers=2, n_heads=2 ===
  -> train_loss=0.4602 | val_loss=3.9500 | seq_acc=0.0090 | BLEU-4=0.1625 | time=6.2s | params=1,068,023

=== Config: n_layers=2, n_heads=4 ===
  -> train_loss=0.3552 | val_loss=3.8865 | seq_acc=0.0000 | BLEU-4=0.1681 | time=7.0s | params=1,068,023

=== Config: n_layers=4, n_heads=2 ===
  -> train_loss=1.1233 | val_loss=4.0237 | seq_acc=0.0090 | BLEU-4=0.1388 | time=11.6s | params=1,730,551

=== Config: n_layers=4, n_heads=4 ===
  -> train_loss=1.0621 | val_loss=3.9797 | seq_acc=0.0000 | BLEU-4=0.1329 | time=11.5s | params=1,730,551


In [ ]:
import pandas as pd

sweep_df = pd.DataFrame(sweep_results).sort_values(["n_layers", "n_heads"]).reset_index(drop=True)
sweep_df


,n_layers,n_heads,final_train_loss,final_val_loss,seq_accuracy,bleu4,training_time_sec,num_params,approx_flops
0,1,2,0.634991,4.252041,0.009009,0.143445,4.945651,736759,6706944
1,1,4,0.488400,4.299303,0.009009,0.148473,4.469023,736759,6706944
2,2,2,0.460238,3.949994,0.009009,0.162507,6.206213,1068023,13413888
3,2,4,0.355166,3.886456,0.000000,0.168119,7.029251,1068023,13413888
4,4,2,1.123273,4.023705,0.009009,0.138787,11.649032,1730551,26827776
5,4,4,1.062128,3.979737,0.000000,0.132901,11.512344,1730551,26827776


In [ ]:
best_row = max(sweep_results, key=lambda r: r["bleu4"])
best_key = (best_row["n_layers"], best_row["n_heads"])
best_model, best_examples = trained_models[best_key]
print(f"Best config: n_layers={best_key[0]}, n_heads={best_key[1]} "
      f"(val BLEU-4={best_row['bleu4']:.4f}, seq_acc={best_row['seq_accuracy']:.4f})\n")

for ex in best_examples:
    print(f"SRC (French)     : {ex['source']}")
    print(f"REF (English)    : {ex['reference']}")
    print(f"PRED (English)   : {ex['prediction']}")
    print(f"Exact match: {ex['exact_match']} | BLEU-4: {ex['bleu']:.4f}")
    print("-" * 60)


Best config: n_layers=2, n_heads=4 (val BLEU-4=0.1681, seq_acc=0.0000)

SRC (French)     : il répond immédiatement à tous les e mails des clients
REF (English)    : he replies to all customer emails immediately
PRED (English)   : he answers the questions
Exact match: False | BLEU-4: 0.0288
------------------------------------------------------------
SRC (French)     : nous travaillons au bureau
REF (English)    : we work in the office
PRED (English)   : we are planning to the desk on the beach
Exact match: False | BLEU-4: 0.0306
------------------------------------------------------------
SRC (French)     : je veux une grande part de gâteau au chocolat
REF (English)    : i want a large slice of chocolate cake
PRED (English)   : i want a large cruise ship the distance
Exact match: False | BLEU-4: 0.3457
------------------------------------------------------------
SRC (French)     : ils parlent souvent d ' histoire
REF (English)    : they often talk about history
PRED (English)   : they 